In [1]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("E:/IDEAL")
META_DIR = BASE_DIR / "metadata"
AGG_DIR = BASE_DIR / "processed/aggregated_metadata"
AGG_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------
# 1) home.csv (core)
# -----------------------
home = pd.read_csv(META_DIR / "home.csv")

# κρατάμε ΜΟΝΟ τα πεδία που κλειδώσαμε
home = home[[
    "homeid",
    "residents",
    "hometype",
    "build_era",
    "location",
    "urban_rural_class",
    "income_band",
    "occupancy"
]]

# -----------------------
# 2) aggregated tables
# -----------------------
room_agg = pd.read_csv(AGG_DIR / "room_aggregated.csv")
appliance_agg = pd.read_csv(AGG_DIR / "appliance_aggregated.csv")
person_agg = pd.read_csv(AGG_DIR / "person_aggregated.csv")

# -----------------------
# 3) Merge όλα σε 1 πίνακα (1 row ανά homeid)
# -----------------------
meta = home.merge(room_agg, on="homeid", how="left") \
           .merge(appliance_agg, on="homeid", how="left") \
           .merge(person_agg, on="homeid", how="left")

# -----------------------
# 4) Έλεγχοι & fill (για σιγουριά)
# -----------------------
# Αν κάποιο σπίτι δεν έχει appliances/person/rooms, οι μετρήσεις θα είναι NaN.
# Για counts -> 0 έχει νόημα.
count_cols = [
    "num_rooms",
    "num_external_windows",
    "num_external_walls",
    "num_appliances",
    "num_electric_appliances",
    "num_gas_appliances",
    "num_children",
    "num_adults",
    "num_elderly",
    "num_working_adults",
    "num_full_time_workers",
    "num_part_time_workers",
    "num_males",
    "num_females",
]
for c in count_cols:
    if c in meta.columns:
        meta[c] = meta[c].fillna(0).astype(int)

# Για εμβαδά (αν λείπουν) τα αφήνουμε NaN (δεν βάζουμε 0 γιατί θα είναι παραπλανητικό)
# meta["total_floor_area_m2"] και meta["avg_room_area_m2"] μένουν όπως είναι

# -----------------------
# 5) Save
# -----------------------
out_path = AGG_DIR / "home_metadata.csv"
meta.to_csv(out_path, index=False)

print("Saved:", out_path.resolve())
print("Rows (homes):", len(meta))
print("Unique homeid:", meta["homeid"].nunique())
meta.head()

Saved: E:\IDEAL\processed\aggregated_metadata\home_metadata.csv
Rows (homes): 255
Unique homeid: 255


,homeid,residents,hometype,build_era,location,urban_rural_class,income_band,occupancy,total_floor_area_m2,num_rooms,...,num_electric_appliances,num_gas_appliances,num_children,num_adults,num_elderly,num_working_adults,num_full_time_workers,num_part_time_workers,num_males,num_females
0,47,2,flat,1900-1918,Edinburgh,1,Missing,multiple,42.7,5,...,6,8,0,2,0,2,1,0,1,1
1,59,2,flat,1900-1918,Edinburgh,1,"£90,000 or more",multiple,76.0,6,...,6,4,1,2,0,2,1,0,1,1
2,61,2,house_or_bungalow,1919-1930,Edinburgh,1,"£48,600 to £53,999",multiple,68.5,9,...,8,9,0,2,0,2,0,0,1,1
3,62,2,flat,1850-1899,Edinburgh,1,"£43,200 to £48,599",multiple,81.5,7,...,8,13,0,2,0,2,0,0,1,1
4,64,4,flat,Before 1850,Edinburgh,1,"£66,000 to £77,999",multiple,65.0,7,...,10,3,2,2,0,2,1,0,2,2
